# Project Description - Automatic Defects Classification for Steel Defects in Manufacturing

**Introduction** <br>
This is a project of defects classification for steel defects on the GC10-DET surface dataset using transfer learning, which the model is pre-trained on ImageNet dataset. A few models such as VGG16, Xception, InceptionResNetV2 and ResNet152V2 will be used for training the dataset and provide comparison between them. 

**Outline** <br>
This project will have the presentation outlined as follows,
1. Dataset visualization
2. Model training and validation
3. Model testing
4. Results evaluation
5. Model prediction
6. Conclusion and acknowledgements

In [ ]:
# Import necessary libraries

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import random
from matplotlib.patches import Rectangle
from lxml import etree

# Enable Matplotlib backend for displaying static images
%matplotlib inline

In [ ]:
# Subfolders for the class labels

import os
os.listdir("/kaggle/input/defects-class-and-location/images/images")

In [ ]:
# Annotations of the class labels

os.listdir("/kaggle/input/defects-class-and-location/label/label")

In [ ]:
# Read all the images and count the number of image paths in the training subfolders

image_path = glob.glob("/kaggle/input/defects-class-and-location/images/images/*/*.jpg")
len(image_path)

In [ ]:
# Read all the annotation files and count the number of annotations in the training label folder

xmls_path = glob.glob("/kaggle/input/defects-class-and-location/label/label/*.xml")
len(xmls_path)

In [ ]:
# Sort the annotation and image files in ascending order

xmls_path.sort(key = lambda x:x.split("/")[-1].split(".xml")[0])
image_path.sort(key = lambda x:x.split("/")[-1].split(".jpg")[0])

In [ ]:
# Extract the annotation filename without file extension

xmls_train = [path.split("/")[-1].split(".")[0] for path in xmls_path]
xmls_train[:5]

In [ ]:
# Retrieve the image filename to match the annotation file counts
# Ensure unique counts of training images with the annotations

imgs_train = [img for img in image_path if (img.split("/")[-1].split)(".jpg")[0] in xmls_train]
imgs_train[:5]

In [ ]:
len(imgs_train),len(xmls_path)

In [ ]:
# Extract label names as DataFrame column

labels = [label.split("/")[-2] for label in imgs_train]
labels[:5]

In [ ]:
labels = pd.DataFrame(labels, columns = ["Defect Type"])
labels

In [ ]:
# One-hot encoding for multiple classes
from sklearn.preprocessing import LabelBinarizer

# Obtain training labels without duplication
Class = labels["Defect Type"].unique()
# Store data values in key:value pairs with Python dictionaries
Class_dict = dict(zip(Class, range(1,len(Class) + 1)))
labels["Class"] = labels["Defect Type"].apply(lambda x: Class_dict[x])

lb = LabelBinarizer()
# Fit label binarizer
lb.fit(list(Class_dict.values()))
# Convert multi-class labels to binary labels (belong or does not belong to the class)
transformed_labels = lb.transform(labels["Class"])
y_bin_labels = []  

for i in range(transformed_labels.shape[1]):
    y_bin_labels.append("Class" + str(i))
    labels["Class" + str(i + 1)] = transformed_labels[:, i]

In [ ]:
Class_dict

In [ ]:
labels.drop("Class", axis = 1, inplace = True)
labels.drop("Defect Type", axis = 1, inplace = True)
labels.head()

In [ ]:
# Function to parse and extract information from annotation files

def to_labels(path):
    # Read the annotation file
    xml = open("{}".format(path)).read()                         
    sel = etree.HTML(xml)
    # Obtain the image width
    width = int(sel.xpath("//size/width/text()")[0])
    # Obtain the image height
    height = int(sel.xpath("//size/height/text()")[0])  
    # Extract the bounding box coordinates
    xmin = int(sel.xpath("//bndbox/xmin/text()")[0])
    xmax = int(sel.xpath("//bndbox/xmax/text()")[0])
    ymin = int(sel.xpath("//bndbox/ymin/text()")[0])
    ymax = int(sel.xpath("//bndbox/ymax/text()")[0])
    # Return the relative coordinates
    return [xmin/width, ymin/height, xmax/width, ymax/height]

In [ ]:
# Display the relative bounding box coordinates

coors = [to_labels(path) for path in xmls_path]
coors[:5]

In [ ]:
# Set four coordinate points as outputs

xmin, ymin, xmax, ymax = list(zip(*coors))

# Convert to Numpy array
xmin = np.array(xmin)
ymin = np.array(ymin)
xmax = np.array(xmax)
ymax = np.array(ymax)
label = np.array(labels.values)

In [ ]:
# Creates a Dataset whose elements are slices of the given tensors
# Slicing a 1D tensor produces scalar tensor elements

labels_dataset = tf.data.Dataset.from_tensor_slices((xmin, ymin, xmax, ymax, label))
labels_dataset

In [ ]:
# Load the image from image path

def load_image(path):
    image = tf.io.read_file(path)                           
    image = tf.image.decode_jpeg(image,3)               
    image = tf.image.resize(image,[224,224])               
    image = tf.cast(image,tf.float32)  
    image = image / 255
    return image

In [ ]:
# Build the dataset

dataset = tf.data.Dataset.from_tensor_slices(imgs_train)
dataset = dataset.map(load_image)

dataset_label = tf.data.Dataset.zip((dataset, labels_dataset))
dataset_label

In [ ]:
# Shuffle the images and extract the images by the defined batch size

batch_size = 32
dataset_label = dataset_label.repeat().shuffle(500).batch(batch_size)

# Creates a Dataset that prefetches elements from this dataset
# Most dataset input pipelines should end with a call to prefetch
# This allows later elements to be prepared while the current element is being processed
# This often improves latency and throughput, at the cost of using additional memory to store prefetched elements

dataset_label = dataset_label.prefetch(tf.data.experimental.AUTOTUNE)

In [ ]:
# Split the dataset into 80% for training and 20% for testing

train_count = int(len(imgs_train) * 0.8)
test_count = int(len(imgs_train) * 0.2)
train_count, test_count

In [ ]:
train_dataset = dataset_label.skip(test_count)
test_dataset = dataset_label.take(test_count)

In [ ]:
class_dict = {v:k for k,v in Class_dict.items()}
class_dict

In [ ]:
# Display a sample training image with its label

for img, label in train_dataset.take(1):
    plt.imshow(keras.preprocessing.image.array_to_img(img[0]))     
    out1, out2, out3, out4, out5 = label                            
    xmin, ymin, xmax, ymax = out1[0].numpy()*224, out2[0].numpy()*224, out3[0].numpy()*224, out4[0].numpy()*224
    rect = Rectangle((xmin,ymin),(xmax - xmin),(ymax - ymin), fill = False, color = "r")  
    ax = plt.gca()                      
    ax.axes.add_patch(rect)   
    pred_imglist = []
    pred_imglist.append(class_dict[np.argmax(out5[0])+1])
    plt.title(pred_imglist)
    plt.show()

**i. VGG16**

In [ ]:
# Import VGG16 model with pre-trained weights from ImageNet

base_vgg16 = tf.keras.applications.VGG16(weights = "imagenet",
                                         include_top = False,
                                         input_shape = (224,224,3),
                                         pooling = 'avg')

In [ ]:
# Train the base model with fully-connected layers

base_vgg16.trainable = True

In [ ]:
# Define the base model with fully-connected layers
inputs = keras.Input(shape = (224,224,3))
x = base_vgg16(inputs)

x1 = keras.layers.Dense(1024, activation = "relu")(x)
x1 = keras.layers.Dense(512, activation = "relu")(x1)
out1 = keras.layers.Dense(1, name = "xmin")(x1)
out2 = keras.layers.Dense(1, name = "ymin")(x1)
out3 = keras.layers.Dense(1, name = "xmax")(x1)
out4 = keras.layers.Dense(1, name = "ymax")(x1)

x2 = keras.layers.Dense(1024,activation = "relu")(x)
x2 = keras.layers.Dropout(0.5)(x2)
x2 = keras.layers.Dense(512,activation = "relu")(x2)
out_class = keras.layers.Dense(10,activation = "softmax", name = "class")(x2)

out = [out1, out2, out3, out4, out_class]

vgg16 = keras.models.Model(inputs = inputs, outputs = out)
vgg16.summary()

In [ ]:
# Compile the model with optimizer and loss functions

vgg16.compile(keras.optimizers.Adam(0.0005),
              loss = {"xmin": "mse",
                      "ymin": "mse",
                      "xmax": "mse",
                      "ymax": "mse",
                      "class": "categorical_crossentropy"},
              metrics = ["mae","acc"])

In [ ]:
# Reduce learning rate

lr_reduce = keras.callbacks.ReduceLROnPlateau("val_loss", patience = 5, factor = 0.5, min_lr = 1e-6)

In [ ]:
# Start training the model

history = vgg16.fit(train_dataset,
                    steps_per_epoch = train_count//batch_size,
                    epochs = 200,
                    validation_data = test_dataset,
                    validation_steps = test_count//batch_size,
                    callbacks = [lr_reduce])

In [ ]:
# Visualize the results after training

def plot_history(history):                
    hist = pd.DataFrame(history.history)           
    hist["epoch"] = history.epoch
    
    plt.figure()                                     
    plt.xlabel("Epoch")
    plt.ylabel("Mean Squared Error (MSE)")               
    plt.plot(hist["epoch"], hist["loss"], label = "training Loss")
    plt.plot(hist["epoch"], hist["val_loss"], label = "validation Loss")                           
    plt.legend()
    
    plt.figure()                                      
    plt.xlabel("Epoch")
    plt.ylabel("Validation Mean Absolute Error (MAE)")               
    plt.plot(hist["epoch"], hist["val_xmin_mae"], label = "xmin_MAE")
    plt.plot(hist["epoch"], hist["val_ymin_mae"], label = "ymin_MAE")
    plt.plot(hist["epoch"], hist["val_xmax_mae"], label = "xmax_MAE")
    plt.plot(hist["epoch"], hist["val_ymax_mae"], label = "ymax_MAE")
    plt.legend()      
    
    plt.figure()                                      
    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy for Class")               
    plt.plot(hist["epoch"],hist["val_class_acc"], label = "class_accuracy")
    
    plt.show()
    
plot_history(history)  

In [ ]:
results = vgg16.evaluate(test_dataset)

In [ ]:
print("MAE of xmin value in test set:{}".format(results[6]))
print("MAE of ymin value in test set:{}".format(results[8]))
print("MAE of xmax value in test set:{}".format(results[10]))
print("MAE of ymax value in test set:{}".format(results[12]))
print("Testing accuracy of predicted label:{}".format(results[15]))

In [ ]:
vgg16.save("vgg16.h5")

In [ ]:
plt.figure(figsize = (10, 24))
for img, _ in test_dataset.take(1):
    out1, out2, out3, out4, label = vgg16.predict(img)
    for i in range(3):
        plt.subplot(3, 1, i + 1)            
        plt.imshow(keras.preprocessing.image.array_to_img(img[i]))    
        pred_imglist = []
        pred_imglist.append(class_dict[np.argmax(out5[i]) + 1])
        plt.title(pred_imglist)
        xmin, ymin, xmax, ymax = out1[i]*224, out2[i]*224, out3[i]*224, out4[i]*224
        rect = Rectangle((xmin,ymin), (xmax - xmin), (ymax - ymin), fill = False, color = "r") 
        ax = plt.gca()                   
        ax.axes.add_patch(rect)

In [ ]:
# Append all the true labels into a list

true_labels = []

for _, label in test_dataset:    
    out1, out2, out3, out4, out5 = label                              
    true_labels.append(class_dict[np.argmax(out5) + 1])

In [ ]:
true_labels

In [ ]:
test_labels = []

for img, label in test_dataset:
    out1, out2, out3, out4, out5 = label
    label = vgg16.predict(img)
    test_labels.append(class_dict[np.argmax(out5) + 1])

In [ ]:
test_labels

In [ ]:
target_names = ['cresent_gap', 'crease', 'silk_spot', 'water_spot', 'welding_line', 
                'inclusion', 'oil_spot', 'waist_folding', 'rolled_pit', 'punching_hole']

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(true_labels, test_labels, target_names = target_names))

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(true_labels, test_labels, labels = target_names)

**ii. Xception**

In [ ]:
# Import Xception model with pre-trained weights from ImageNet

base_xception = tf.keras.applications.Xception(weights = "imagenet",
                                               include_top = False,
                                               input_shape = (224,224,3),
                                               pooling = 'avg')

In [ ]:
# Train the base model with fully-connected layers

base_xception.trainable = True

In [ ]:
# Define the base model with fully-connected layers
inputs = keras.Input(shape = (224,224,3))
x = base_xception(inputs)

x1 = keras.layers.Dense(1024, activation = "relu")(x)
x1 = keras.layers.Dense(512, activation = "relu")(x1)
out1 = keras.layers.Dense(1, name = "xmin")(x1)
out2 = keras.layers.Dense(1, name = "ymin")(x1)
out3 = keras.layers.Dense(1, name = "xmax")(x1)
out4 = keras.layers.Dense(1, name = "ymax")(x1)

x2 = keras.layers.Dense(1024, activation = "relu")(x)
x2 = keras.layers.Dropout(0.5)(x2)
x2 = keras.layers.Dense(512, activation = "relu")(x2)
out_class = keras.layers.Dense(10, activation = "softmax", name = "class")(x2)

out = [out1, out2, out3, out4, out_class]

xception = keras.models.Model(inputs = inputs, outputs = out)
xception.summary()

In [ ]:
# Compile the model with optimizer and loss functions

xception.compile(keras.optimizers.Adam(0.0005),
                 loss = {"xmin": "mse",
                         "ymin": "mse",
                         "xmax": "mse",
                         "ymax": "mse",
                         "class": "categorical_crossentropy"},
                 metrics = ["mae","acc"])

In [ ]:
# Start training the model

history = xception.fit(train_dataset,
                       steps_per_epoch = train_count//batch_size,
                       epochs = 200,
                       validation_data = test_dataset,
                       validation_steps = test_count//batch_size,
                       callbacks = [lr_reduce])

In [ ]:
plot_history(history)  

In [ ]:
results = xception.evaluate(test_dataset)

In [ ]:
print("MAE of xmin value in test set:{}".format(results[6]))
print("MAE of ymin value in test set:{}".format(results[8]))
print("MAE of xmax value in test set:{}".format(results[10]))
print("MAE of ymax value in test set:{}".format(results[12]))
print("Testing accuracy of predicted label:{}".format(results[15]))

In [ ]:
xception.save("xception.h5")

In [ ]:
plt.figure(figsize = (10, 24))
for img, _ in test_dataset.take(1):
    out1, out2, out3, out4, label = xception.predict(img)
    for i in range(3):
        plt.subplot(3, 1, i + 1)            
        plt.imshow(keras.preprocessing.image.array_to_img(img[i]))    
        pred_imglist = []
        pred_imglist.append(class_dict[np.argmax(out5[i]) + 1])
        plt.title(pred_imglist)
        xmin, ymin, xmax, ymax = out1[i]*224, out2[i]*224, out3[i]*224, out4[i]*224
        rect = Rectangle((xmin,ymin), (xmax - xmin), (ymax - ymin), fill = False, color = "r") 
        ax = plt.gca()                   
        ax.axes.add_patch(rect)

In [ ]:
test_labels = []

for img, label in test_dataset:
    out1, out2, out3, out4, out5 = label
    label = xception.predict(img)
    test_labels.append(class_dict[np.argmax(out5) + 1])

In [ ]:
test_labels

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(true_labels, test_labels, target_names = target_names))

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(true_labels, test_labels, labels = target_names)

**iii. InceptionResNetV2**

In [ ]:
# Import InceptionResNetV2 model with pre-trained weights from ImageNet

base_inceptionresnetv2 = tf.keras.applications.InceptionResNetV2(weights = "imagenet",
                                                                 include_top = False,
                                                                 input_shape = (224,224,3),
                                                                 pooling = 'avg')

In [ ]:
# Train the base model with fully-connected layers

base_inceptionresnetv2.trainable = True

In [ ]:
# Define the base model with fully-connected layers
inputs = keras.Input(shape = (224,224,3))
x = base_inceptionresnetv2(inputs)

x1 = keras.layers.Dense(1024, activation = "relu")(x)
x1 = keras.layers.Dense(512, activation = "relu")(x1)
out1 = keras.layers.Dense(1, name = "xmin")(x1)
out2 = keras.layers.Dense(1, name = "ymin")(x1)
out3 = keras.layers.Dense(1, name = "xmax")(x1)
out4 = keras.layers.Dense(1, name = "ymax")(x1)

x2 = keras.layers.Dense(1024, activation = "relu")(x)
x2 = keras.layers.Dropout(0.5)(x2)
x2 = keras.layers.Dense(512, activation = "relu")(x2)
out_class = keras.layers.Dense(10, activation = "softmax", name = "class")(x2)

out = [out1, out2, out3, out4, out_class]

inceptionresnetv2 = keras.models.Model(inputs = inputs, outputs = out)
inceptionresnetv2.summary()

In [ ]:
# Compile the model with optimizer and loss functions

inceptionresnetv2.compile(keras.optimizers.Adam(0.0005),
                          loss = {"xmin": "mse",
                                  "ymin": "mse",
                                  "xmax": "mse",
                                  "ymax": "mse",
                                  "class": "categorical_crossentropy"},
                          metrics = ["mae","acc"])

In [ ]:
# Start training the model

history = inceptionresnetv2.fit(train_dataset,
                                steps_per_epoch = train_count//batch_size,
                                epochs = 200,
                                validation_data = test_dataset,
                                validation_steps = test_count//batch_size,
                                callbacks = [lr_reduce])

In [ ]:
plot_history(history) 

In [ ]:
results = inceptionresnetv2.evaluate(test_dataset)

In [ ]:
print("MAE of xmin value in test set:{}".format(results[6]))
print("MAE of ymin value in test set:{}".format(results[8]))
print("MAE of xmax value in test set:{}".format(results[10]))
print("MAE of ymax value in test set:{}".format(results[12]))
print("Testing accuracy of predicted label:{}".format(results[15]))

In [ ]:
inceptionresnetv2.save("inceptionresnetv2.h5")

In [ ]:
plt.figure(figsize = (10, 24))
for img, _ in test_dataset.take(1):
    out1, out2, out3, out4, label = inceptionresnetv2.predict(img)
    for i in range(3):
        plt.subplot(3, 1, i + 1)            
        plt.imshow(keras.preprocessing.image.array_to_img(img[i]))    
        pred_imglist = []
        pred_imglist.append(class_dict[np.argmax(out5[i]) + 1])
        plt.title(pred_imglist)
        xmin, ymin, xmax, ymax = out1[i]*224, out2[i]*224, out3[i]*224, out4[i]*224
        rect = Rectangle((xmin,ymin), (xmax - xmin), (ymax - ymin), fill = False, color = "r") 
        ax = plt.gca()                   
        ax.axes.add_patch(rect)

In [ ]:
test_labels = []

for img, label in test_dataset:
    out1, out2, out3, out4, out5 = label
    label = inceptionresnetv2.predict(img)
    test_labels.append(class_dict[np.argmax(out5) + 1])

In [ ]:
test_labels

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(true_labels, test_labels, target_names = target_names))

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(true_labels, test_labels, labels = target_names)

**iv. ResNet152V2**

In [ ]:
# Import InceptionResNetV2 model with pre-trained weights from ImageNet

base_resnet152v2 = tf.keras.applications.ResNet152V2(weights = "imagenet",
                                                     include_top = False,
                                                     input_shape = (224,224,3),
                                                     pooling = 'avg')

In [ ]:
# Train the base model with fully-connected layers

base_resnet152v2.trainable = True

In [ ]:
# Define the base model with fully-connected layers
inputs = keras.Input(shape = (224,224,3))
x = base_resnet152v2(inputs)

x1 = keras.layers.Dense(1024, activation = "relu")(x)
x1 = keras.layers.Dense(512, activation = "relu")(x1)
out1 = keras.layers.Dense(1, name = "xmin")(x1)
out2 = keras.layers.Dense(1, name = "ymin")(x1)
out3 = keras.layers.Dense(1, name = "xmax")(x1)
out4 = keras.layers.Dense(1, name = "ymax")(x1)

x2 = keras.layers.Dense(1024, activation = "relu")(x)
x2 = keras.layers.Dropout(0.5)(x2)
x2 = keras.layers.Dense(512, activation = "relu")(x2)
out_class = keras.layers.Dense(10, activation = "softmax", name = "class")(x2)

out = [out1, out2, out3, out4, out_class]

resnet152v2 = keras.models.Model(inputs = inputs, outputs = out)
resnet152v2.summary()

In [ ]:
# Compile the model with optimizer and loss functions

resnet152v2.compile(keras.optimizers.Adam(0.0005),
                    loss = {"xmin": "mse",
                            "ymin": "mse",
                            "xmax": "mse",
                            "ymax": "mse",
                            "class": "categorical_crossentropy"},
                    metrics = ["mae","acc"])

In [ ]:
# Start training the model

history = resnet152v2.fit(train_dataset,
                          steps_per_epoch = train_count//batch_size,
                          epochs = 200,
                          validation_data = test_dataset,
                          validation_steps = test_count//batch_size,
                          callbacks = [lr_reduce])

In [ ]:
plot_history(history)

In [ ]:
results = resnet152v2.evaluate(test_dataset)

In [ ]:
print("MAE of xmin value in test set:{}".format(results[6]))
print("MAE of ymin value in test set:{}".format(results[8]))
print("MAE of xmax value in test set:{}".format(results[10]))
print("MAE of ymax value in test set:{}".format(results[12]))
print("Testing accuracy of predicted label:{}".format(results[15]))

In [ ]:
resnet152v2.save("resnet152v2.h5")

In [ ]:
plt.figure(figsize = (10, 24))
for img, _ in test_dataset.take(1):
    out1, out2, out3, out4, label = resnet152v2.predict(img)
    for i in range(3):
        plt.subplot(3, 1, i + 1)            
        plt.imshow(keras.preprocessing.image.array_to_img(img[i]))    
        pred_imglist = []
        pred_imglist.append(class_dict[np.argmax(out5[i]) + 1])
        plt.title(pred_imglist)
        xmin, ymin, xmax, ymax = out1[i]*224, out2[i]*224, out3[i]*224, out4[i]*224
        rect = Rectangle((xmin,ymin), (xmax - xmin), (ymax - ymin), fill = False, color = "r") 
        ax = plt.gca()                   
        ax.axes.add_patch(rect)

In [ ]:
test_labels = []

for img, label in test_dataset:
    out1, out2, out3, out4, out5 = label
    label = resnet152v2.predict(img)
    test_labels.append(class_dict[np.argmax(out5) + 1])

In [ ]:
test_labels

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(true_labels, test_labels, target_names = target_names))

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(true_labels, test_labels, labels = target_names)

**References:**
1. https://www.fatalerrors.org/a/tensorflow-learning-notes-no.10.html
2. https://valueml.com/steel-surface-inspection-in-keras-python/
3. https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices
4. https://github.com/tensorflow/tensorflow/issues/14857
5. https://stackoverflow.com/questions/56613155/tensorflow-tf-data-autotune
6. https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch
7. https://www.tensorflow.org/api_docs/python/tf/cast
8. https://realpython.com/python-map-function/
9. https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html
10. https://stackoverflow.com/questions/50473381/scikit-learns-labelbinarizer-vs-onehotencoder